In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
import torch
from torch.utils.data import DataLoader, Dataset
import numpy as np
import pandas as pd

In [3]:
# Define the dataset
class MyDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("DeepChem/ChemBERTa-77M-MLM")

In [5]:
# Load the data

resp = pd.read_csv("KIM_pubchem_hepg2_chemberta_embeddings.csv")
resp

,Unnamed: 0,CAS_no.,compound,name,molecular_formula,canonical_smiles,canonicalize,EC50(uM),EC50(mg/L),EC10(uM),...,emb_374,emb_375,emb_376,emb_377,emb_378,emb_379,emb_380,emb_381,emb_382,emb_383
0,0,120-83-2,"2,4-DCP","2,4-dichlorophenol",C6H4Cl2O,C1=CC(=C(C=C1Cl)Cl)O,Clc1ccc(c(c1)Cl)O,682.20,111.20,328.600,...,-0.367104,0.023312,0.062783,0.131439,0.039820,0.031435,-0.041275,-0.162000,-0.606179,-0.360052
1,1,583-78-8,"2,5-DCP","2,5-dichlorophenol",C6H4Cl2O,C1=CC(=C(C=C1Cl)O)Cl,Clc1ccc(c(c1)O)Cl,867.80,141.50,305.300,...,-0.396367,0.060893,0.097783,0.170509,-0.183774,0.054095,-0.056161,-0.240935,-0.729859,-0.415152
2,2,94-13-3,pPB,Propyl paraben,C10H12O3,CCCOC(=O)C1=CC=C(C=C1)O,CCCOC(=O)c1ccc(cc1)O,742.40,133.80,269.800,...,-0.104164,0.169499,0.229696,0.296413,-0.099188,-0.412370,-0.357306,-0.185529,-0.618633,-0.182249
3,3,94-26-8,bPB,Butyl paraben,C11H14O3,CCCCOC(=O)C1=CC=C(C=C1)O,CCCCOC(=O)c1ccc(cc1)O,346.20,67.23,191.700,...,-0.187763,0.019691,0.275738,0.214475,-0.201018,-0.148579,-0.435028,-0.267638,-0.517660,-0.089917
4,4,131-57-7,BP-3,Oxybenzone,C14H12O3,COC1=CC(=C(C=C1)C(=O)C2=CC=CC=C2)O,COc1ccc(c(c1)O)C(=O)c1ccccc1,1166.00,266.20,227.100,...,-0.027794,0.296864,0.084269,0.011656,-0.208684,-0.366786,-0.081210,-0.369623,-0.244868,-0.344598
5,5,3380-34-5,TCS,Triclosan,C12H7Cl3O2,C1=CC(=C(C=C1Cl)O)OC2=C(C=C(C=C2)Cl)Cl,Clc1ccc(c(c1)O)Oc1ccc(cc1Cl)Cl,19.29,5.59,5.041,...,-0.180233,0.192036,0.041122,0.141791,0.088449,0.354120,-0.215963,-0.084222,-0.798773,-0.371715
6,6,7758-95-4,Pb,Lead chloride,Cl2Pb,Cl[Pb]Cl,Cl[Pb]Cl,917.10,255.10,380.000,...,-0.000837,-0.027256,0.120374,-0.696434,0.116556,-0.117984,-0.293882,-0.320630,-0.525666,-0.241756
7,7,7758-98-7,Cu,Copper sulphate,CuSO4,[O-]S(=O)(=O)[O-].[Cu+2],[O-]S(=O)(=O)[O-],60.45,15.09,5.349,...,-0.173878,0.067683,0.044728,-0.121903,-0.236400,0.075067,-0.229852,-0.220079,0.131329,-0.473357
8,8,10102-18-8,Se,Sodium selenite,Na2O3Se,[O-][Se](=O)[O-].[Na+].[Na+],[O-][Se](=O)[O-],84.59,14.63,4.213,...,-0.189837,-0.177703,0.114857,0.002861,-0.347643,0.162264,-0.137043,-0.059859,0.210664,-0.548381
9,9,654054-66-7,Cd,Cadmium chloride hydrate,CdCl2H2O,O.Cl[Cd]Cl,Cl[Cd]Cl,5.56,1.02,0.841,...,0.125153,0.128021,0.076943,0.248730,0.127234,-0.362770,-0.379875,-0.070035,-0.780885,-0.171606


In [6]:
resp["metal"] = [0,0,0,0,0,0,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0]
resp

,Unnamed: 0,CAS_no.,compound,name,molecular_formula,canonical_smiles,canonicalize,EC50(uM),EC50(mg/L),EC10(uM),...,emb_374,emb_375,emb_376,emb_377,emb_378,emb_379,emb_380,emb_381,emb_382,emb_383
0,0,120-83-2,"2,4-DCP","2,4-dichlorophenol",C6H4Cl2O,C1=CC(=C(C=C1Cl)Cl)O,Clc1ccc(c(c1)Cl)O,682.20,111.20,328.600,...,-0.367104,0.023312,0.062783,0.131439,0.039820,0.031435,-0.041275,-0.162000,-0.606179,-0.360052
1,1,583-78-8,"2,5-DCP","2,5-dichlorophenol",C6H4Cl2O,C1=CC(=C(C=C1Cl)O)Cl,Clc1ccc(c(c1)O)Cl,867.80,141.50,305.300,...,-0.396367,0.060893,0.097783,0.170509,-0.183774,0.054095,-0.056161,-0.240935,-0.729859,-0.415152
2,2,94-13-3,pPB,Propyl paraben,C10H12O3,CCCOC(=O)C1=CC=C(C=C1)O,CCCOC(=O)c1ccc(cc1)O,742.40,133.80,269.800,...,-0.104164,0.169499,0.229696,0.296413,-0.099188,-0.412370,-0.357306,-0.185529,-0.618633,-0.182249
3,3,94-26-8,bPB,Butyl paraben,C11H14O3,CCCCOC(=O)C1=CC=C(C=C1)O,CCCCOC(=O)c1ccc(cc1)O,346.20,67.23,191.700,...,-0.187763,0.019691,0.275738,0.214475,-0.201018,-0.148579,-0.435028,-0.267638,-0.517660,-0.089917
4,4,131-57-7,BP-3,Oxybenzone,C14H12O3,COC1=CC(=C(C=C1)C(=O)C2=CC=CC=C2)O,COc1ccc(c(c1)O)C(=O)c1ccccc1,1166.00,266.20,227.100,...,-0.027794,0.296864,0.084269,0.011656,-0.208684,-0.366786,-0.081210,-0.369623,-0.244868,-0.344598
5,5,3380-34-5,TCS,Triclosan,C12H7Cl3O2,C1=CC(=C(C=C1Cl)O)OC2=C(C=C(C=C2)Cl)Cl,Clc1ccc(c(c1)O)Oc1ccc(cc1Cl)Cl,19.29,5.59,5.041,...,-0.180233,0.192036,0.041122,0.141791,0.088449,0.354120,-0.215963,-0.084222,-0.798773,-0.371715
6,6,7758-95-4,Pb,Lead chloride,Cl2Pb,Cl[Pb]Cl,Cl[Pb]Cl,917.10,255.10,380.000,...,-0.000837,-0.027256,0.120374,-0.696434,0.116556,-0.117984,-0.293882,-0.320630,-0.525666,-0.241756
7,7,7758-98-7,Cu,Copper sulphate,CuSO4,[O-]S(=O)(=O)[O-].[Cu+2],[O-]S(=O)(=O)[O-],60.45,15.09,5.349,...,-0.173878,0.067683,0.044728,-0.121903,-0.236400,0.075067,-0.229852,-0.220079,0.131329,-0.473357
8,8,10102-18-8,Se,Sodium selenite,Na2O3Se,[O-][Se](=O)[O-].[Na+].[Na+],[O-][Se](=O)[O-],84.59,14.63,4.213,...,-0.189837,-0.177703,0.114857,0.002861,-0.347643,0.162264,-0.137043,-0.059859,0.210664,-0.548381
9,9,654054-66-7,Cd,Cadmium chloride hydrate,CdCl2H2O,O.Cl[Cd]Cl,Cl[Cd]Cl,5.56,1.02,0.841,...,0.125153,0.128021,0.076943,0.248730,0.127234,-0.362770,-0.379875,-0.070035,-0.780885,-0.171606


In [7]:
train_smiles = list(resp["canonicalize"][:7].values) + list(resp["canonical_smiles"][7:9].values) + list(resp["canonicalize"][9:13].values) + [resp["canonical_smiles"].iloc[13].replace("O.O.O.O.O.O.O.", "")] + list(resp["canonicalize"][14:].values)
train_smiles

['Clc1ccc(c(c1)Cl)O',
 'Clc1ccc(c(c1)O)Cl',
 'CCCOC(=O)c1ccc(cc1)O',
 'CCCCOC(=O)c1ccc(cc1)O',
 'COc1ccc(c(c1)O)C(=O)c1ccccc1',
 'Clc1ccc(c(c1)O)Oc1ccc(cc1Cl)Cl',
 'Cl[Pb]Cl',
 '[O-]S(=O)(=O)[O-].[Cu+2]',
 '[O-][Se](=O)[O-].[Na+].[Na+]',
 'Cl[Cd]Cl',
 'Cl[Sb](Cl)Cl',
 'Cl[Co]Cl',
 'Cl[Ni]Cl',
 '[O-]S(=O)(=O)[O-].[Zn+2]',
 'C[Hg]Cl',
 'OC(=O)C(C(C(C(C(C(C(F)(F)F)(F)F)(F)F)(F)F)(F)F)(F)F)(F)F',
 'OC(=O)C(C(C(C(C(C(C(C(F)(F)F)(F)F)(F)F)(F)F)(F)F)(F)F)(F)F)(F)F',
 'C(C(C(C(F)(F)S(=O)(=O)O)(F)F)(F)F)(C(C(F)(F)F)(F)F)(F)F',
 'CCCCOP(=O)(OCCCC)OCCCC',
 'C1=CC=C(C=C1)OP(=O)(OC2=CC=CC=C2)OC3=CC=CC=C3',
 'ClCC(OP(=O)(OC(CCl)C)OC(CCl)C)C',
 'CCCCC(CC)COP(=O)(OC1=CC=CC=C1)OC2=CC=CC=C2',
 'CCCCOCCOP(=O)(OCCOCCCC)OCCOCCCC',
 'ClCC(OP(=O)(OC(CCl)CCl)OC(CCl)CCl)CCl',
 'CCCCC(COP(=O)(OCC(CCCC)CC)OCC(CCCC)CC)CC']

In [ ]:
train_labels = (resp['EC50(uM)'] <= 100).astype(int)
train_labels

In [11]:
# Tokenize the training and validation data
train_encodings = tokenizer(train_smiles, truncation=True, padding=True)

In [12]:
# Create training and validation datasets
train_dataset = MyDataset(train_encodings, train_labels)

In [ ]:
# Load the pre-trained ChemBERTa model with a classification head
model = AutoModelForSequenceClassification.from_pretrained("DeepChem/ChemBERTa-77M-MLM", num_labels=2)

In [14]:
device = torch.device("cuda")
model.to(device)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(600, 384, padding_idx=1)
      (position_embeddings): Embedding(515, 384, padding_idx=1)
      (token_type_embeddings): Embedding(1, 384)
      (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.144, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-2): 3 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=384, out_features=384, bias=True)
              (key): Linear(in_features=384, out_features=384, bias=True)
              (value): Linear(in_features=384, out_features=384, bias=True)
              (dropout): Dropout(p=0.109, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=384, out_features=384, bias=True)
             

# Compute embeddings

In [15]:
inputs = tokenizer(train_smiles, 
                   return_tensors="pt",
                   padding="max_length",
#                   max_length=512,
                   truncation=True)

In [16]:
inputs = {k: v.to(model.device) for k,v in inputs.items()}

In [17]:
class ChemBERTaEmbeddingExtractor(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids, attention_mask=None):
        # Assuming 'model' has a method to get the transformer outputs directly
        # The actual method to access the transformer layer's output might vary
        outputs = self.model(input_ids, attention_mask=attention_mask)

        # Extracting the last hidden state
        last_hidden_state = outputs[0]  # Shape: [batch_size, sequence_length, hidden_size]

        # Getting embeddings, e.g., for the [CLS] token
        embeddings = last_hidden_state[:, 0, :]

        return embeddings

In [18]:
# Create an extractor instance
extractor = ChemBERTaEmbeddingExtractor(model)

In [19]:
inputs

{'input_ids': tensor([[12, 16, 15,  ...,  0,  0,  0],
         [12, 16, 15,  ...,  0,  0,  0],
         [12, 16, 16,  ...,  0,  0,  0],
         ...,
         [12, 16, 16,  ...,  0,  0,  0],
         [12, 16, 16,  ...,  0,  0,  0],
         [12, 16, 16,  ...,  0,  0,  0]], device='cuda:0'),
 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         ...,
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0]], device='cuda:0')}

In [20]:
# Prepare your input_ids and attention_mask
input_ids = inputs["input_ids"]
attention_mask = inputs["attention_mask"]

In [21]:
import torch
from torch.utils.data import DataLoader, TensorDataset

# Assuming `input_ids` and `attention_mask` are your inputs prepared as PyTorch tensors
dataset = TensorDataset(input_ids, attention_mask)
batch_size = 32  # Choose a batch size that fits your GPU memory
loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)  # Shuffle=False for sequential data

# Assuming embedding_extractor is your previously defined model wrapper
model.eval()  # Set the model to evaluation mode

all_embeddings = []  # To store embeddings from all batches

with torch.no_grad():  # Disables gradient calculation to save memory
    for batch_input_ids, batch_attention_mask in loader:
        # Process each batch through the model
        outputs = model.roberta(batch_input_ids, attention_mask=batch_attention_mask)

        # Access the last hidden state directly
        # This assumes you're interested in the embeddings from the last hidden state
        last_hidden_state = outputs.last_hidden_state

        # Example: Extract embeddings of the first token ([CLS] token)
        batch_embeddings = last_hidden_state[:, 0, :]  # Shape: (batch_size, hidden_size)

        # Move the extracted embeddings to CPU to save GPU memory
        all_embeddings.append(batch_embeddings.cpu())

# Concatenate all batch embeddings to get a single tensor
embeddings = torch.cat(all_embeddings, dim=0)

# Now `embeddings` contains the embeddings for your entire dataset, processed in batches

In [23]:
embeddings.shape

torch.Size([25, 384])

In [28]:
embeddings

tensor([[-0.2040, -0.4105,  0.3070,  ...,  0.0430, -0.5692, -0.5579],
        [-0.2223, -0.3474,  0.2380,  ..., -0.0323, -0.6679, -0.3820],
        [-0.1663,  0.0477,  0.2448,  ..., -0.0456, -0.1605, -0.0344],
        ...,
        [-0.4717,  0.0515, -0.0445,  ...,  0.2087,  0.0681,  0.2561],
        [ 0.0140, -0.2013, -0.3322,  ...,  0.0862,  0.1874, -0.5036],
        [-0.2785, -0.0804, -0.0180,  ...,  0.1847,  0.2644,  0.0542]])

In [25]:
new_emb = embeddings.reshape(25, -1)
new_emb.shape

torch.Size([25, 384])

In [ ]:
final_embs = pd.DataFrame(embeddings, columns=[ "emb_{}".format(i) for i in range(0,384) ])
final_embs

In [ ]:
meta = resp.iloc[:, 1:15]
meta

In [ ]:
KIM = pd.merge(meta, final_embs, left_index=True, right_index=True,)
KIM

In [ ]:
KIM.to_csv("KIM_pubchem_hepg2_chemberta_embeddings_PRETRAINED.csv")